In [1]:
%pip install -U openai python-dotenv langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu docx2txt

import os
from pathlib import Path

from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader, Docx2txtLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\nguye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\nguye\AppData\Local\Temp\ipykernel_3412\2974213046.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, Docx2txtLoader


In [2]:
DATA_FOLDER = "data"

EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 3

TEMPERATURE = 0
MAX_TOKENS = 300

In [3]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Không tìm thấy OPENAI_API_KEY trong file .env")


def load_documents(folder_path: str):
    documents = []
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Không tìm thấy folder: {folder_path}")

    for file_path in folder.rglob("*"):
        if file_path.suffix.lower() == ".txt":
            documents.extend(
                TextLoader(
                    str(file_path),
                    encoding="utf-8"
                ).load()
            )

        elif file_path.suffix.lower() == ".docx":
            documents.extend(
                Docx2txtLoader(str(file_path)).load()
            )

    if not documents:
        raise ValueError(
            "Folder data không có file .txt hoặc .docx."
        )

    return documents


documents = load_documents(DATA_FOLDER)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)


def run(question: str) -> str:
    question = question.strip()

    if not question:
        return "Bạn chưa nhập câu hỏi."

    relevant_documents = vector_store.similarity_search(
        query=question,
        k=min(TOP_K, len(chunks))
    )

    context = "\n\n".join(
        document.page_content
        for document in relevant_documents
    )

    prompt = f"""
Bạn là chatbot trả lời dựa trên tài liệu được cung cấp.

Quy định:
- Chỉ sử dụng thông tin trong phần TÀI LIỆU.
- Không tự bổ sung thông tin bên ngoài.
- Nếu không có thông tin, trả lời:
  "Tôi không tìm thấy thông tin này trong tài liệu."
- Trả lời bằng tiếng Việt, rõ ràng và ngắn gọn.

TÀI LIỆU:
{context}

CÂU HỎI:
{question}
"""

    response = llm.invoke(prompt)

    return response.content

In [4]:
question = "Có mấy loại thẻ tín dụng?"

answer = run(question)

print("Câu hỏi:")
print(question)

print("\nCâu trả lời:")
print(answer)

Câu hỏi:
Có mấy loại thẻ tín dụng?

Câu trả lời:
Tôi không tìm thấy thông tin này trong tài liệu.
